# Fine-Tuning LFM3-VL-2B to Read Receipts into JSON with TRL (SFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_vl_sft_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

This notebook fine-tunes [LFM3-VL-2B](https://huggingface.co/LiquidAI/LFM3-VL-2B) to read photos of receipts into clean JSON. We use Supervised Fine-Tuning (SFT) with [TRL](https://github.com/huggingface/trl) and LoRA, and measure field-level accuracy before and after training on receipts the model never saw.

Extracting structured data from images is a typical on-device task: the photo stays on the phone, and a small model that reliably fills your schema is faster and cheaper than a large general one.

## 🎯 What you'll find

- **The task**: a receipt photo goes in, a JSON object that follows a fixed schema comes out
- **Reasoning**: the model first reads the receipt line by line in its reasoning, then writes the JSON
- **LoRA + SFT**: train the language model on the reasoning and the answer, with the vision encoder frozen
- **Evaluation**: valid JSON rate, field accuracy, and exact match, on the held-out test receipts

# 📦 Installation & Setup

In [ ]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

In [ ]:
import json
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-VL-2B is released in bfloat16. GPUs without bfloat16 support, like the T4, load it in float16 instead: the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Load the model from Transformers 🤗

LFM3-VL-2B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`.

The processor turns each picture into vision tokens, about one token per 32×32 pixels. We limit pictures to 1280×1280 pixels (about 1,600 tokens) with the processor's `size` (`longest_edge` and `shortest_edge` are pixel counts): the text stays readable, and training and generation stay fast. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "LiquidAI/LFM3-VL-2B"

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, size={"shortest_edge": 256 * 256, "longest_edge": 1280 * 1280})
processor.tokenizer.padding_side = "left"

print("🧠 Loading model...")
model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

# 📚 Load the receipt dataset

[CORD](https://huggingface.co/datasets/naver-clova-ix/cord-v2) holds 1,000 photos of receipts from shops and restaurants in Indonesia, with the printed text of each line and its meaning (item name, count, price, total, ...). It has three splits: `train` (800), `validation` (100) and `test` (100).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("naver-clova-ix/cord-v2")
print(dataset)

example = dataset["train"][1]
print(f"\n🧾 Picture: {example['image'].size[0]}×{example['image'].size[1]} pixels")
print(f"\n🎯 Labels:\n{json.dumps(json.loads(example['ground_truth'])['gt_parse'], indent=2, ensure_ascii=False)[:800]}")
example["image"].resize((example["image"].width * 500 // example["image"].height, 500))

## Define the schema

We turn CORD's labels into a fixed schema: the purchased items in printed order, then the totals and the payment. Every value is copied exactly as printed (`"24,000"`, `"2 x"`), and a field that is not on the receipt is `null`. Item modifiers printed under an item (a size, a topping, the sugar level) are left out.

A few receipts have two values for one field (for example two cash payments). They don't fit the schema, so we drop them.

In [ ]:
FIELDS = {  # schema key -> CORD label
    "subtotal": ("sub_total", "subtotal_price"),
    "discount": ("sub_total", "discount_price"),
    "service": ("sub_total", "service_price"),
    "tax": ("sub_total", "tax_price"),
    "total": ("total", "total_price"),
    "cash": ("total", "cashprice"),
    "change": ("total", "changeprice"),
    "card": ("total", "creditcardprice"),
}

def to_schema(labels):
    """The extraction of one receipt in our schema, or None when a field has several values."""
    menu = labels.get("menu", [])
    menu = menu if isinstance(menu, list) else [menu]
    items = [{"name": item.get("nm"), "quantity": item.get("cnt"), "price": item.get("price")} for item in menu]
    totals = {key: labels.get(group, {}).get(label) if isinstance(labels.get(group), dict) else None for key, (group, label) in FIELDS.items()}
    extraction = {"items": items, **totals}
    values = [v for item in items for v in item.values()] + list(totals.values())
    if not all(v is None or isinstance(v, str) for v in values) or any(item["name"] is None for item in items):
        return None
    return extraction

print(json.dumps(to_schema(json.loads(example["ground_truth"])["gt_parse"]), indent=2, ensure_ascii=False))

## Write the reasoning

LFM3-VL-2B thinks before it answers, and we keep it that way. Asked about a receipt, the original model already starts its reasoning by reading the lines of the receipt. CORD gives us the printed text of every line, so we turn it into a short reasoning in the same spirit: the lines of the receipt, top to bottom. The model learns to read first, then fill the schema from what it read, and its reasoning stays short.

In [ ]:
from collections import defaultdict

def receipt_lines(ground_truth):
    """The printed lines of a receipt, top to bottom, each with its words left to right."""
    rows = defaultdict(list)
    for line in ground_truth["valid_line"]:
        for word in line["words"]:
            quad = word["quad"]
            rows[word["row_id"]].append((min(quad["x1"], quad["x4"]), (quad["y1"] + quad["y3"]) / 2, word["text"]))
    rows = sorted(rows.values(), key=lambda words: sum(y for _, y, _ in words) / len(words))
    return [" ".join(text for _, _, text in sorted(words)) for words in rows]

def reasoning(ground_truth):
    return "The receipt reads, line by line:\n" + "\n".join(receipt_lines(ground_truth)) + "\n\nNow I fill the schema from these lines."

print(reasoning(json.loads(example["ground_truth"])))

## Build the conversations

The system prompt holds the schema and the extraction rules, and the user message holds the photo. In the training conversations, the assistant message holds the reasoning (in `reasoning_content`) and the JSON answer, on one line. The chat template writes the reasoning between `<think>` and `</think>`.

The conversations are built from the labels only. The photos stay in their `image` column, and TRL puts each photo in front of the user text. This way `map` does not decode and write again 770 large photos, which would need more RAM than a free Colab runtime has.

In [ ]:
SYSTEM_PROMPT = """Extract the receipt in the picture into a JSON object with exactly these keys, in this order:

{"items": [{"name": string, "quantity": string or null, "price": string or null}], "subtotal": string or null, "discount": string or null, "service": string or null, "tax": string or null, "total": string or null, "cash": string or null, "change": string or null, "card": string or null}

Rules:
1. Output exactly one JSON object, on one line. No prose, no code fences.
2. Copy every value exactly as printed: same digits, separators, symbols and spelling. Do not convert or compute values.
3. `items` lists the purchased items in printed order. Leave out modifiers printed under an item (size, topping, sugar level).
4. `quantity` is the count printed for the item, `price` the amount printed on the item's line.
5. `subtotal` is the amount before tax and service, `discount` the total discount, `service` the service charge, `tax` the tax, `total` the amount to pay, `cash` the cash given, `change` the change returned, `card` the amount paid by card.
6. A field that is not printed on the receipt is null."""

def build_messages(image):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Extract this receipt."}]},
    ]

def to_conversation(ground_truth):
    ground_truth = json.loads(ground_truth)
    extraction = to_schema(ground_truth["gt_parse"])
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Extract this receipt."},  # TRL puts the picture before this text
            {"role": "assistant", "content": json.dumps(extraction, ensure_ascii=False), "reasoning_content": reasoning(ground_truth)},
        ],
    }

def keep(ground_truth):
    return to_schema(json.loads(ground_truth)["gt_parse"]) is not None

dataset = dataset.filter(keep, input_columns="ground_truth")
train_dataset = dataset["train"].map(to_conversation, input_columns="ground_truth", remove_columns="ground_truth")
print(f"🧾 {len(train_dataset)} training receipts, {len(dataset['test'])} test receipts")
print(json.dumps(train_dataset[1]["messages"][2], indent=2, ensure_ascii=False)[:1200])

# 📏 Evaluate the model before training

We generate an extraction for each test receipt, with the sampling settings recommended in the model card (they are the checkpoint's generation defaults), and compare it with the gold JSON:

- **Valid JSON**: the answer after the reasoning parses as a JSON object (we tolerate a Markdown code fence around it, even though the rules ask for none).
- **Field accuracy**: the share of fields with the right value. CORD's labels space words inconsistently (`"JASMINE MT ( L )"`), so strings are compared without case and spaces. `null` must stay `null`.
- **Exact match**: every field is right.
- **Output tokens**: the average length of the reasoning and the answer, which sets the latency on a phone.

The original model has never seen our schema, so it gets the same instructions as the fine-tuned one.

In [ ]:
model.generation_config

In [ ]:
from tqdm.auto import tqdm

def parse_json(text):
    """The JSON object in a model's answer, tolerating a Markdown code fence around it, or None."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0]
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def flatten(value, path=""):
    """Leaves of a JSON value as {path: value}, e.g. {"items[0].price": "24,000"}."""
    if isinstance(value, dict):
        return {k: v for key, child in value.items() for k, v in flatten(child, f"{path}.{key}" if path else key).items()}
    if isinstance(value, list):
        return {k: v for i, child in enumerate(value) for k, v in flatten(child, f"{path}[{i}]").items()} or {path: []}
    return {path: value}

def same(a, b):
    if isinstance(a, str) and isinstance(b, str):
        return "".join(a.split()).casefold() == "".join(b.split()).casefold()
    return a == b

def score(prediction, gold):
    """Field accuracy of one prediction: matching fields / fields in the gold or the prediction."""
    gold_fields, predicted_fields = flatten(gold), flatten(prediction)
    paths = set(gold_fields) | set(predicted_fields)
    if not paths:
        return 1.0
    return sum(p in gold_fields and p in predicted_fields and same(gold_fields[p], predicted_fields[p]) for p in paths) / len(paths)

def answer(text):
    """The answer after the reasoning, or None when the reasoning never ends."""
    return text.split("</think>", 1)[1] if "</think>" in text else None

@torch.no_grad()
def generate(model, images, max_new_tokens=4096):
    """The reasoning and answer of the model for each picture, as text, and their lengths in tokens."""
    inputs = processor.apply_chat_template(
        [build_messages(image) for image in images], add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", processor_kwargs={"padding": True},
    ).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)[:, inputs["input_ids"].shape[1]:]
    lengths = (outputs != processor.tokenizer.pad_token_id).sum(dim=1).tolist()
    return processor.batch_decode(outputs, skip_special_tokens=True), lengths

def evaluate(model, batch_size=16):
    model.eval()
    examples = dataset["test"]
    valid, accuracy, exact, tokens = 0, 0.0, 0, 0
    for start in tqdm(range(0, len(examples), batch_size), desc="Evaluating"):
        batch = examples.select(range(start, min(start + batch_size, len(examples))))
        texts, lengths = generate(model, batch["image"])
        tokens += sum(lengths)
        for text, example in zip(texts, batch):
            gold = to_schema(json.loads(example["ground_truth"])["gt_parse"])
            prediction = parse_json(answer(text) or "")
            if prediction is None:
                continue  # invalid JSON scores 0 on every metric
            valid += 1
            field_accuracy = score(prediction, gold)
            accuracy += field_accuracy
            exact += field_accuracy == 1.0
    n = len(examples)
    return {"valid_json": valid / n, "field_accuracy": accuracy / n, "exact_match": exact / n, "tokens": tokens / n}

def show(r, title):
    print(f"{title}: valid JSON {r['valid_json']:.1%} | field accuracy {r['field_accuracy']:.1%} | exact match {r['exact_match']:.1%} | {r['tokens']:.0f} output tokens")

torch.manual_seed(0)
baseline = evaluate(model, batch_size=32)
show(baseline, "📏 Before training")

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

## Wrap the model with PEFT

In LFM3's language model, we adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice. The vision encoder uses some of the same names, so we select the language model's layers with a regular expression and leave the vision encoder frozen: it already sees receipts well, the model only has to learn what to write.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # optional: add "|gate_proj" to the group for LFM3's per-head attention gate
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|w1|w2|w3)",
    task_type="CAUSAL_LM",
)

## Launch training

With `assistant_only_loss=True`, `SFTTrainer` computes the loss on the assistant message only: the model learns to write the reasoning and the JSON, not to reproduce the system prompt. The chat template marks the assistant message for TRL with `{% generation %}`.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./LFM3-VL-2B-receipts",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # <- effective batch size of 16 receipts
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    max_length=None,  # <- never cut a conversation: the vision tokens of a picture must stay whole
    assistant_only_loss=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

print("🚀 Starting SFT training...")
trainer.train()
print("🎉 SFT training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# 📊 Evaluate the fine-tuned model

In [ ]:
torch.manual_seed(0)
finetuned = evaluate(trainer.model, batch_size=32)
show(baseline, "📏 Before training")
show(finetuned, "📏 After training")

# 🔍 Original vs fine-tuned model

To see what changed, we run the original model and the fine-tuned model on the same test receipts. With LoRA, we don't need to load the original model again: `disable_adapter()` temporarily turns the adapter off.

For each field, we show the gold value and both extractions, with ✅ when the value is right.

In [ ]:
from IPython.display import display

def cell(fields, path, gold_fields):
    if fields is None:
        return "❌ invalid JSON"
    if path not in fields:
        return "❌ (missing)"
    mark = "✅" if path in gold_fields and same(fields[path], gold_fields[path]) else "❌"
    return f"{mark} {json.dumps(fields[path], ensure_ascii=False)}"

trainer.model.eval()
for example in dataset["test"].select([3, 17, 42]):
    with trainer.model.disable_adapter():
        (original_text,), _ = generate(trainer.model, [example["image"]])
    (finetuned_text,), _ = generate(trainer.model, [example["image"]])
    gold_fields = flatten(to_schema(json.loads(example["ground_truth"])["gt_parse"]))
    original, finetuned_extraction = parse_json(answer(original_text) or ""), parse_json(answer(finetuned_text) or "")
    original_fields = flatten(original) if original is not None else None
    finetuned_fields = flatten(finetuned_extraction) if finetuned_extraction is not None else None
    display(example["image"].resize((example["image"].width * 400 // example["image"].height, 400)))
    print(f"🧠 Fine-tuned reasoning:\n{finetuned_text.split('</think>')[0][:600]}\n")
    print(f"{'field':<20} {'gold':<24} {'original':<30} {'fine-tuned'}")
    for path, gold_value in gold_fields.items():
        print(f"{path[:19]:<20} {json.dumps(gold_value, ensure_ascii=False)[:23]:<24} {cell(original_fields, path, gold_fields)[:29]:<30} {cell(finetuned_fields, path, gold_fields)[:36]}")

And the summary over all the test receipts:

In [ ]:
print(f"{'':<16}{'original':>10}{'fine-tuned':>12}")
for metric in ["valid_json", "field_accuracy", "exact_match"]:
    print(f"{metric:<16}{baseline[metric]:>10.1%}{finetuned[metric]:>12.1%}")
print(f"{'output tokens':<16}{baseline['tokens']:>10.0f}{finetuned['tokens']:>12.0f}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3-VL's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-VL-2B-receipts-lora")

output_dir = "./LFM3-VL-2B-receipts-merged"
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model to the Hugging Face Hub, uncomment:
# merged_model.push_to_hub("your-username/LFM3-VL-2B-receipts")
# processor.push_to_hub("your-username/LFM3-VL-2B-receipts")

# 🚀 Next steps

- **Your own documents**: write a schema and label a few hundred photos. Include photos where fields are missing, so the model learns to return `null` instead of guessing.
- **Your own reasoning**: if your labels have no line-by-line text, generate reasoning with the original model, and keep only the conversations whose answer matches your labels.
- **Constrained decoding**: at inference, a JSON-schema grammar (supported by llama.cpp and vLLM) guarantees valid JSON on top of the fine-tuned accuracy.